# ハザードデータ ジオメトリ調査（hazard_geometry_audit）

公式配布のハザードデータのうち、`sheltermatch.ipynb` が「空・不正なジオメトリ」として区域判定から
除外しているレコードが、実際には何なのかを可視化するための **調査専用Notebook** です。

## このNotebookの位置付け（重要）

- **sheltermatch本体ではありません。** 避難所候補の算出や、職員が使う結果CSVの作成は行いません。
- **公式ハザードデータの品質確認用** です。
- **本体のデータを自動修復するものではありません。** `shapely.make_valid()` はこのNotebookの中だけで
  試し、`sheltermatch.ipynb` の読込仕様・判定ロジックは一切変更しません。
- **調査結果を見てから、本体へ修復処理を導入するかどうかを判断します。** このNotebookは判断材料を
  出すところまでを担当し、「本体へ導入すべき」と結論づけることはしません。

現行 `sheltermatch.ipynb` が不正なジオメトリを削除している方式を、あらかじめ誤りと決めつけては
いません。削除されている1,000件規模のレコードが、区域として意味のあるものなのか、それとも
区域判定に使えない断片なのかを確かめることが目的です。

## 調査のきっかけ

Google Colab上で、公式配布データ11ZIP（津波LEVEL1〜7 / A33土砂災害 / 高潮 / A31a 2種類）を投入した
実行で、有効ポリゴン181,070件を読み込めた一方、合計1,354件が

```python
geometry.notna() & geometry.is_valid
```

の条件を満たさず「空・不正なジオメトリ」として除外されました（level1〜7で計1,048件、
A31a_10で145件、A31a_20で161件）。

## 明らかにすること

1. 除外されたジオメトリの実態（null / empty / `is_valid=False` の内訳と、その理由）
2. `shapely.make_valid()` でどの程度Polygon/MultiPolygonへ復元できるか
3. 復元した場合に、ハザード判定結果（要支援者地点・候補避難所地点・両者を結ぶ直線）が
   実際に変わり得るか

## 使い方

1. 「ランタイム → すべてのセルを実行」
2. 公式ハザードZIP（複数選択可）をアップロードする
3. 要支援者CSV（`resident_id,address,latitude,longitude,geocode_status` の5列）をアップロードする
4. 最後のサマリセルの出力を確認する

公式データを外部サイトから自動ダウンロードすることはありません。アップロードされたファイルだけを
使います。

## 個人情報の取り扱い

要支援者CSVには個人情報が含まれ得ます。**利用を許可された環境でのみ扱ってください。**
このNotebookに実データをハードコードしないでください。また、実際の要支援者データ・出力を
このリポジトリへコミットしないでください。


In [ ]:
# ===== 1. 実行環境準備 =====
# Google Colabに標準で入っていないライブラリをインストールします（初回のみ数十秒かかることがあります）。
%pip install -q geopy geopandas shapely

import io
import re
import tempfile
import warnings
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from geopy.distance import geodesic

import geopandas as gpd
import shapely
from shapely.geometry import LineString, Point
from pyproj import Geod

from google.colab import files

# 区域判定に使えるジオメトリ種別（sheltermatch.ipynbと同じ）。
POLYGON_TYPES = ["Polygon", "MultiPolygon"]

# 面積の目安を平方メートルで出すために使う（WGS84楕円体上の面積）。
GEOD = Geod(ellps="WGS84")

print("ライブラリの読み込みが完了しました。")
print(f"  geopandas {gpd.__version__} / shapely {shapely.__version__}")


In [ ]:
# ===== 2. 公式ハザードZIPアップロード・全レコード読込 =====
# 公式配布のハザードデータ（A31a / A33 / 津波LEVEL1〜7 / 高潮）をまとめてアップロードします。
# 外部サイトからの自動ダウンロードは行いません。
#
# ZIPの展開・レイヤー探索・ハザード種別の自動判定・CRS処理は、sheltermatch.ipynb と同じ考え方で
# 行います（CP932ファイル名の復元、バックスラッシュ区切りの正規化、パストラバーサル対策、
# GeoJSON/Shapefileの再帰探索、EPSG:4326への統一）。
#
# ただし本体と違い、**ジオメトリの絞り込みは一切行わず、元レコードをすべて読み込みます**。
# 何が除外されているのかを数えることが目的のため、除外前の状態が必要になるためです。


def _decode_zip_entry_name(member):
    """ZIPエントリのファイル名を復号する。UTF-8フラグ（bit 11）が立っていないエントリは
    CP437として解釈されるため、CP932（Shift-JIS系）の日本語ファイル名を含む配布ZIP
    （沖縄県公式データ等）では文字化けする。その場合はCP932として再解釈する。"""
    if member.flag_bits & 0x800:
        return member.filename
    try:
        return member.filename.encode("cp437").decode("cp932")
    except (UnicodeDecodeError, UnicodeEncodeError):
        return member.filename


def safe_extract_zip(zip_bytes, extract_dir):
    """ZIPをextract_dirへ安全に展開する。エントリ名は文字コード復元→パス区切り('\\'→'/')正規化
    の順で処理する。正規化後のパスが展開先の外に出るエントリが1件でもあれば、展開を一切行わずに
    例外を送出する（パストラバーサル対策。全エントリを先に検査してから展開する）。"""
    extract_dir_abs = Path(extract_dir).resolve()
    with zipfile.ZipFile(io.BytesIO(zip_bytes)) as zip_file:
        resolved_members = []
        for member in zip_file.infolist():
            name = _decode_zip_entry_name(member).replace("\\", "/")
            is_dir_entry = name.endswith("/")
            member_path = (extract_dir_abs / name).resolve()
            if member_path != extract_dir_abs and extract_dir_abs not in member_path.parents:
                raise RuntimeError(f"ZIP内に不正なパスが含まれているため展開を中止しました: {name}")
            resolved_members.append((member, member_path, is_dir_entry))

        for member, member_path, is_dir_entry in resolved_members:
            if is_dir_entry:
                member_path.mkdir(parents=True, exist_ok=True)
                continue
            member_path.parent.mkdir(parents=True, exist_ok=True)
            with zip_file.open(member) as source, open(member_path, "wb") as target:
                target.write(source.read())


def find_shapefile_layers(extract_dir):
    """配下の.shpを再帰探索し、同じディレクトリに.shx/.dbfが揃っているものだけを返す。
    戻り値は (有効な.shpパス, 揃っていないため除外した件数, 発見した.shp総数)。"""
    shp_paths = sorted(extract_dir.rglob("*.shp"))
    valid_paths = []
    missing_companion_count = 0
    for shp_path in shp_paths:
        siblings = {p.name.lower() for p in shp_path.parent.iterdir()}
        if f"{shp_path.stem.lower()}.shx" in siblings and f"{shp_path.stem.lower()}.dbf" in siblings:
            valid_paths.append(shp_path)
        else:
            missing_companion_count += 1
    return valid_paths, missing_companion_count, len(shp_paths)


# --- ハザード種別の自動判定（sheltermatch.ipynbと同じ規則） ---

A33_PHENOMENON_LABELS = {"1": "急傾斜地の崩壊", "2": "土石流", "3": "地滑り"}
A33_ZONE_LABELS = {
    "1": "土砂災害警戒区域(指定済)",
    "2": "土砂災害特別警戒区域(指定済)",
    "3": "土砂災害警戒区域(指定前)",
    "4": "土砂災害特別警戒区域(指定前)",
}


def normalize_code_value(value):
    """公式データのコード値を、対応表を引くための文字列へ揃える（1 / "1" / 1.0 → "1"）。"""
    if pd.isna(value):
        return None
    text = str(value).strip()
    try:
        number = float(text)
        if number.is_integer():
            return str(int(number))
    except (TypeError, ValueError):
        pass
    return text


def derive_feature_hazard_types(gdf, hazard_type):
    """レイヤーの属性から、行ごとのhazard_typeを組み立てる（A33の公式属性、津波等の『分類』属性）。"""
    if "A33_001" in gdf.columns and "A33_002" in gdf.columns:
        def build_a33_hazard_type(row):
            phenomenon = A33_PHENOMENON_LABELS.get(
                normalize_code_value(row["A33_001"]), normalize_code_value(row["A33_001"]))
            zone = A33_ZONE_LABELS.get(
                normalize_code_value(row["A33_002"]), normalize_code_value(row["A33_002"]))
            parts = [part for part in (phenomenon, zone) if part]
            return f"{hazard_type}:{':'.join(parts)}" if parts else hazard_type

        return gdf.apply(build_a33_hazard_type, axis=1)

    if "分類" in gdf.columns:
        return gdf["分類"].apply(
            lambda value: f"{hazard_type}:{value}" if pd.notna(value) and str(value).strip() else hazard_type
        )

    return pd.Series([hazard_type] * len(gdf), index=gdf.index, dtype="object")


def _detect_a31a_hazard_type(filename):
    match = re.match(r"^A31a-\d+_\d+_(10|20)(?=[_.])", filename)
    if not match:
        return None
    return {"10": "洪水（洪水予報河川・水位周知河川）", "20": "洪水（その他の河川）"}[match.group(1)]


def _detect_a33_hazard_type(filename):
    return "土砂災害" if re.match(r"^A33-\d+_\d+_GEOJSON", filename) else None


def _detect_tsunami_level_hazard_type(filename):
    return "津波" if re.match(r"^level[1-7](?=[_.])", filename, re.IGNORECASE) else None


def _detect_takashio_hazard_type(filename):
    return "高潮" if re.match(r"^\d+_takasiosinnsuisoutei_", filename) else None


KNOWN_HAZARD_FILENAME_DETECTORS = [
    _detect_a31a_hazard_type,
    _detect_a33_hazard_type,
    _detect_tsunami_level_hazard_type,
    _detect_takashio_hazard_type,
]


def detect_hazard_type(filename):
    """既知の公式データ配布ファイル名パターンからハザード種別名を自動判定する（不明ならNone）。"""
    for detector in KNOWN_HAZARD_FILENAME_DETECTORS:
        hazard_type = detector(filename)
        if hazard_type is not None:
            return hazard_type
    return None


def _takashio_category_from_stem(stem):
    prefix, sep, _rest = stem.rpartition("_")
    return prefix if sep else stem


def resolve_layer_hazard_type(base_hazard_type, vector_path, extract_dir):
    """ZIP展開先直下のサブフォルダ名をカテゴリとしてhazard_typeへ反映する
    （高潮のみ、ファイル名から市町村名部分を除いた名前を優先する）。"""
    if base_hazard_type == "高潮":
        category = _takashio_category_from_stem(vector_path.stem)
    else:
        rel_parts = vector_path.relative_to(extract_dir).parts
        category = rel_parts[0] if len(rel_parts) > 1 else None
    return f"{base_hazard_type}:{category}" if category else base_hazard_type


# --- レイヤー読込（絞り込みなし） ---


def load_layer_records(source, layer_hazard_type, assume_wgs84_without_crs):
    """1レイヤーを読み込み、hazard_type/geometryの2列へ正規化してEPSG:4326へ統一する。
    CRS処理はsheltermatch.ipynbと同じ（CRSがあれば変換、無ければGeoJSONはWGS84とみなし、
    Shapefileは座標値が経緯度として妥当な範囲のときだけEPSG:4326と推定する）。
    本体と違い、ジオメトリによる絞り込みは行わず元レコードをすべて返す。
    戻り値は (GeoDataFrame, crs_note)。crs_noteは 'assumed_by_bounds' / 'unresolved' / None。"""
    gdf = gpd.read_file(source)

    empty_result = gpd.GeoDataFrame({"hazard_type": [], "geometry": []}, crs="EPSG:4326")
    if len(gdf) == 0:
        return empty_result, None

    crs_note = None
    if gdf.crs is not None:
        if gdf.crs.to_epsg() != 4326:
            gdf = gdf.to_crs(epsg=4326)
    elif assume_wgs84_without_crs:
        gdf = gdf.set_crs(epsg=4326)
    else:
        minx, miny, maxx, maxy = gdf.total_bounds
        if -180 <= minx and maxx <= 180 and -90 <= miny and maxy <= 90:
            gdf = gdf.set_crs(epsg=4326)
            crs_note = "assumed_by_bounds"
        else:
            return empty_result, "unresolved"

    records = gpd.GeoDataFrame(
        {
            "hazard_type": derive_feature_hazard_types(gdf, layer_hazard_type).values,
            "geometry": gdf.geometry.values,
        },
        crs="EPSG:4326",
    )
    return records, crs_note


def load_upload_records(filename, file_bytes, base_hazard_type):
    """1つのアップロード（.geojson または公式配布ZIP）から、全レコードのGeoDataFrameを作る。
    file列にアップロードファイル名、layer列にZIP内のパスを入れ、後から内訳を追えるようにする。"""
    suffix = Path(filename).suffix.lower()
    if suffix not in (".geojson", ".zip"):
        raise ValueError(f"'{filename}' は対応していない形式です。.geojson または .zip を選択してください。")

    layers = []
    skipped = []

    if suffix == ".geojson":
        records, crs_note = load_layer_records(io.BytesIO(file_bytes), base_hazard_type, True)
        if crs_note == "unresolved":
            skipped.append((filename, "座標系を特定できない"))
        else:
            records["file"] = filename
            records["layer"] = filename
            records["hazard_base"] = base_hazard_type
            layers.append(records)
    else:
        with tempfile.TemporaryDirectory(prefix="hazard_audit_") as extract_dir_str:
            extract_dir = Path(extract_dir_str)
            safe_extract_zip(file_bytes, extract_dir)

            geojson_paths = sorted(extract_dir.rglob("*.geojson"))
            shp_paths, missing_companion_count, total_shp_found = find_shapefile_layers(extract_dir)
            if missing_companion_count:
                skipped.append((filename, f".shx/.dbfが揃っていないShapefile {missing_companion_count}件"))
            if not geojson_paths and total_shp_found == 0:
                raise RuntimeError(f"'{filename}' 内に.geojsonまたはShapefile(.shp)が見つかりませんでした。")

            for vector_path, is_geojson in (
                [(p, True) for p in geojson_paths] + [(p, False) for p in shp_paths]
            ):
                layer_hazard_type = resolve_layer_hazard_type(base_hazard_type, vector_path, extract_dir)
                records, crs_note = load_layer_records(vector_path, layer_hazard_type, is_geojson)
                if crs_note == "unresolved":
                    skipped.append((str(vector_path.relative_to(extract_dir)), "座標系を特定できない"))
                    continue
                if len(records) == 0:
                    continue
                records["file"] = filename
                records["layer"] = str(vector_path.relative_to(extract_dir))
                records["hazard_base"] = base_hazard_type
                layers.append(records)

    if not layers:
        return gpd.GeoDataFrame(
            {"hazard_type": [], "geometry": [], "file": [], "layer": [], "hazard_base": []},
            crs="EPSG:4326",
        ), skipped
    return gpd.GeoDataFrame(pd.concat(layers, ignore_index=True), crs="EPSG:4326"), skipped


print("公式ハザードデータ（.zip または .geojson）を選択してください（複数選択可）。")
uploaded_hazards = files.upload()

if not uploaded_hazards:
    raise RuntimeError("ハザードデータがアップロードされませんでした。調査対象のZIPを選択してください。")

upload_frames = []
skipped_layers = []
for hazard_filename, hazard_bytes in uploaded_hazards.items():
    base_hazard_type = detect_hazard_type(hazard_filename)
    if base_hazard_type is not None:
        print(f"'{hazard_filename}' → ハザード種別を自動判定: {base_hazard_type}")
    else:
        base_hazard_type = input(
            f"'{hazard_filename}' のハザード種別を自動判定できませんでした。\n"
            "ハザード種別名を入力してください（例: 洪水, 土砂災害, 津波, 高潮）: "
        ).strip() or hazard_filename

    frame, skipped = load_upload_records(hazard_filename, hazard_bytes, base_hazard_type)
    upload_frames.append(frame)
    skipped_layers.extend(skipped)
    print(f"  レコード数: {len(frame)}件")

hazard_records = gpd.GeoDataFrame(pd.concat(upload_frames, ignore_index=True), crs="EPSG:4326")

print()
print(f"読み込んだ元レコード総数: {len(hazard_records)}件")
print(f"アップロードファイル数: {len(uploaded_hazards)}件 / レイヤー数: {hazard_records['layer'].nunique()}件")
if skipped_layers:
    print("\n読み込めず調査対象外としたレイヤー:")
    for name, reason in skipped_layers:
        print(f"  - {name}: {reason}")


In [ ]:
# ===== 3. ジオメトリ分類・is_valid=Falseの理由集計 =====
# 読み込んだ元レコードを、現行 sheltermatch.ipynb の視点で分類します。
#
#   null          : ジオメトリそのものが無い
#   empty         : 空のジオメトリ（座標を持たない）
#   invalid       : is_valid=False（自己交差等）
#   non_polygon   : 有効だがPolygon/MultiPolygon以外（LineString等。区域判定の対象外）
#   valid_polygon : 有効なPolygon/MultiPolygon（現行本体がそのまま使っているもの）
#
# あわせて、現行本体の絞り込み条件をそのまま適用した結果（kept_by_current）も計算し、
# どの分類が実際に除外されているのかを、決め打ちせず実データで確認します。
#
# 注意: 空のジオメトリが現行条件で残るか除外されるかは、geopandasのバージョンによって
# GeoSeries.notna() の扱いが異なるため変わり得ます。ここでは推測せず、この実行環境での
# 実際の結果を表示します。

geometry = hazard_records.geometry

is_null = geometry.isna()
is_empty = (~is_null) & geometry.is_empty
is_invalid = (~is_null) & (~is_empty) & (~geometry.is_valid)
is_polygonal = geometry.geom_type.isin(POLYGON_TYPES)

hazard_records["category"] = np.select(
    [is_null, is_empty, is_invalid, ~is_polygonal],
    ["null", "empty", "invalid", "non_polygon"],
    default="valid_polygon",
)

# 現行 sheltermatch.ipynb の絞り込み条件をそのまま適用する（load_hazard_layer と同じ式）。
# 空のジオメトリがあると geopandas が GeoSeries.notna() の仕様変更を警告するが、その内容は
# このあと日本語で表示するため、ここでは警告そのものは抑えて出力を読みやすくしている。
with warnings.catch_warnings():
    warnings.filterwarnings("ignore", "GeoSeries.notna", UserWarning)
    hazard_records["kept_by_current"] = (
        geometry.notna() & geometry.is_valid & geometry.geom_type.isin(POLYGON_TYPES)
    )

print(f"geopandas {gpd.__version__} / shapely {shapely.__version__} での結果")
print(f"元レコード総数: {len(hazard_records)}件")
print()
print("分類別の件数と、現行本体での扱い:")
category_summary = (
    hazard_records.groupby("category")
    .agg(件数=("category", "size"), 現行本体が使用=("kept_by_current", "sum"))
    .reindex(["valid_polygon", "null", "empty", "invalid", "non_polygon"])
    .dropna(how="all")
)
category_summary["現行本体が除外"] = category_summary["件数"] - category_summary["現行本体が使用"]
display(category_summary.astype("Int64"))

excluded_total = int((~hazard_records["kept_by_current"]).sum())
print(f"現行本体が区域判定に使用するレコード: {int(hazard_records['kept_by_current'].sum())}件")
print(f"現行本体が除外するレコード          : {excluded_total}件")

if int((hazard_records["category"] == "empty").sum()):
    empty_kept = int(hazard_records.loc[hazard_records["category"] == "empty", "kept_by_current"].sum())
    print(
        f"\n※ 空のジオメトリ {int((hazard_records['category'] == 'empty').sum())}件のうち "
        f"{empty_kept}件は、この環境の geopandas では現行条件を通過しています"
        "（通過しても座標を持たないため、どの地点・直線とも交差しません）。"
    )


def summarize_invalid_reason(geom):
    """is_valid=False の理由を集計しやすい形へ整える。shapelyは
    'Self-intersection[127.6 26.1]' のように座標付きで返すため、座標部分を取り除く
    （理由の文言は決め打ちせず、実データに現れたものをそのまま集計する）。"""
    reason = shapely.is_valid_reason(geom)
    if reason is None:
        return "理由を取得できない"
    return reason.split("[", 1)[0].strip()


invalid_records = hazard_records[hazard_records["category"] == "invalid"].copy()

if len(invalid_records) == 0:
    print("\nis_valid=False のジオメトリはありませんでした。")
else:
    invalid_records["invalid_reason"] = invalid_records.geometry.apply(summarize_invalid_reason)
    hazard_records.loc[invalid_records.index, "invalid_reason"] = invalid_records["invalid_reason"]

    print(f"\nis_valid=False の理由の内訳（{len(invalid_records)}件）:")
    display(
        invalid_records["invalid_reason"]
        .value_counts()
        .rename_axis("invalid_reason")
        .to_frame("件数")
    )

    print("ハザード種別ごとの理由の内訳:")
    display(
        pd.crosstab(
            invalid_records["file"], invalid_records["invalid_reason"], margins=True, margins_name="合計"
        )
    )


In [ ]:
# ===== 4. make_valid()による復元可能性の検証 =====
# is_valid=False のジオメトリへ shapely.make_valid() を適用し、結果がどうなるかを分類します。
#
# ここで行うのは「復元できるかどうかの確認」だけです。make_valid()に成功したからといって、
# 本体で採用してよいという判断にはしません（公式データが意図した区域と一致する保証は無いため）。
#
# GeometryCollectionが返る場合は、中にPolygon/MultiPolygon成分があるかどうかも確認し、
# 調査用にその成分だけを取り出してみます。

def extract_polygonal_parts(geom):
    """ジオメトリからPolygon/MultiPolygon成分だけを取り出す。
    取り出せない場合はNoneを返す。GeometryCollectionの中身を調べるために使う。"""
    if geom is None or geom.is_empty:
        return None
    if geom.geom_type in POLYGON_TYPES:
        return geom
    if geom.geom_type != "GeometryCollection":
        return None

    parts = [
        part for part in geom.geoms
        if part.geom_type in POLYGON_TYPES and not part.is_empty
    ]
    if not parts:
        return None
    return shapely.union_all(parts)


def polygon_area_m2(geom):
    """ポリゴンのおおよその面積を平方メートルで返す（WGS84楕円体上の面積）。
    復元されたのが実体のある区域なのか、面積がほぼ無い断片なのかを見分けるために使う。"""
    if geom is None or geom.is_empty:
        return 0.0
    area, _perimeter = GEOD.geometry_area_perimeter(geom)
    return abs(area)


def audit_make_valid(geom):
    """1件のジオメトリへmake_valid()を適用し、調査結果をまとめて返す。

    repaired_type      : make_valid()の結果のジオメトリ種別（空ならempty）
    repaired_usable    : 区域判定に使えるPolygon/MultiPolygonを取り出せたか
    repaired_from_gc   : その取り出しがGeometryCollectionからの抽出によるものか
    repaired_geometry  : 取り出したPolygon/MultiPolygon（取り出せなければNone）
    repaired_area_m2   : 取り出したポリゴンのおおよその面積
    """
    try:
        repaired = shapely.make_valid(geom)
    except Exception as error:  # make_valid自体が失敗するケースも記録して落とさない
        return {
            "repaired_type": f"make_valid失敗({type(error).__name__})",
            "repaired_usable": False,
            "repaired_from_gc": False,
            "repaired_geometry": None,
            "repaired_area_m2": 0.0,
        }

    repaired_type = "empty" if repaired is None or repaired.is_empty else repaired.geom_type
    polygonal = extract_polygonal_parts(repaired)
    usable = polygonal is not None and polygonal.is_valid and not polygonal.is_empty

    return {
        "repaired_type": repaired_type,
        "repaired_usable": bool(usable),
        "repaired_from_gc": bool(usable and repaired_type == "GeometryCollection"),
        "repaired_geometry": polygonal if usable else None,
        "repaired_area_m2": polygon_area_m2(polygonal) if usable else 0.0,
    }


# 集計セルで使えるよう、調査結果を元のレコード側にも持たせる（invalid以外は既定値のまま）。
hazard_records["repaired_type"] = pd.NA
hazard_records["repaired_usable"] = False
hazard_records["repaired_from_gc"] = False
hazard_records["repaired_area_m2"] = 0.0

if len(invalid_records) == 0:
    make_valid_results = pd.DataFrame(
        columns=["repaired_type", "repaired_usable", "repaired_from_gc",
                 "repaired_geometry", "repaired_area_m2"]
    )
    print("is_valid=False のジオメトリが無いため、make_valid()の検証対象はありません。")
else:
    make_valid_results = pd.DataFrame(
        [audit_make_valid(geom) for geom in invalid_records.geometry],
        index=invalid_records.index,
    )

    for column in ("repaired_type", "repaired_usable", "repaired_from_gc", "repaired_area_m2"):
        hazard_records.loc[make_valid_results.index, column] = make_valid_results[column]

    print(f"make_valid()を適用したレコード: {len(make_valid_results)}件")
    print()
    print("make_valid()の結果のジオメトリ種別:")
    display(
        make_valid_results["repaired_type"]
        .value_counts()
        .rename_axis("repaired_type")
        .to_frame("件数")
    )

    usable_count = int(make_valid_results["repaired_usable"].sum())
    from_gc_count = int(make_valid_results["repaired_from_gc"].sum())
    print(f"Polygon/MultiPolygonとして取り出せた件数: {usable_count}件")
    print(f"  うちGeometryCollectionから成分を抽出した件数: {from_gc_count}件")
    print(f"取り出せなかった件数（区域判定に使えない）: {len(make_valid_results) - usable_count}件")

    gc_results = make_valid_results[make_valid_results["repaired_type"] == "GeometryCollection"]
    if len(gc_results):
        print(
            f"\nGeometryCollectionが返った {len(gc_results)}件のうち、"
            f"Polygon成分があったのは {int(gc_results['repaired_usable'].sum())}件です。"
        )

    if usable_count:
        areas = make_valid_results.loc[make_valid_results["repaired_usable"], "repaired_area_m2"]
        print()
        print("復元されたポリゴンのおおよその面積（平方メートル）:")
        print(f"  合計   : {areas.sum():,.1f}")
        print(f"  最小   : {areas.min():,.4f}")
        print(f"  中央値 : {areas.median():,.4f}")
        print(f"  最大   : {areas.max():,.1f}")
        print(
            "  ※ 面積がほぼ0のものは、区域としての実体が無い断片（自己交差の解消で生じる線状の"
            "かけら等）である可能性があります。"
        )
        print(f"  面積が1平方メートル未満: {int((areas < 1).sum())}件 / {len(areas)}件")


In [ ]:
# ===== 5. ファイル・ハザード種別ごとの復元可能件数の集計 =====
# ここまでの分類結果を、アップロードファイルとハザード種別ごとの表にまとめます。
# 最後の行が全ファイル合計です。

audit_table = (
    hazard_records.assign(
        元レコード数=1,
        現行有効Polygon=hazard_records["kept_by_current"].astype(int),
        null件数=(hazard_records["category"] == "null").astype(int),
        empty件数=(hazard_records["category"] == "empty").astype(int),
        invalid件数=(hazard_records["category"] == "invalid").astype(int),
        Polygon以外件数=(hazard_records["category"] == "non_polygon").astype(int),
        make_valid復元可=hazard_records["repaired_usable"].astype(int),
        GC成分抽出=hazard_records["repaired_from_gc"].astype(int),
    )
    .groupby(["file", "hazard_base"], as_index=False)[
        [
            "元レコード数", "現行有効Polygon", "null件数", "empty件数", "invalid件数",
            "Polygon以外件数", "make_valid復元可", "GC成分抽出",
        ]
    ]
    .sum()
)

# make_validしても区域判定に使えないもの（invalidのうち復元できなかった件数）。
audit_table["復元不能"] = audit_table["invalid件数"] - audit_table["make_valid復元可"]

audit_table = audit_table.sort_values(["hazard_base", "file"]).reset_index(drop=True)

total_row = audit_table.drop(columns=["file", "hazard_base"]).sum()
total_row["file"] = "【合計】"
total_row["hazard_base"] = ""
audit_table_with_total = pd.concat(
    [audit_table, pd.DataFrame([total_row])[audit_table.columns]], ignore_index=True
)

print("ファイル・ハザード種別ごとのジオメトリ内訳")
print("（make_valid復元可 = make_valid()後にPolygon/MultiPolygonとして取り出せた件数。")
print("  GC成分抽出 = そのうちGeometryCollectionから成分を抽出したもので、make_valid復元可の内数）")
display(audit_table_with_total)

print("ハザード種別ごとの合計:")
display(
    audit_table.drop(columns=["file"])
    .groupby("hazard_base", as_index=False)
    .sum()
)


In [ ]:
# ===== 6. 現行方式と修復方式の比較用レイヤー作成 =====
# 判定結果を比べるために、2つのGeoDataFrameを作ります。
#
#   hazard_current  : 現行 sheltermatch.ipynb と同じ条件（有効なPolygon/MultiPolygonのみ）
#   hazard_repaired : hazard_current ＋ make_valid()でPolygon/MultiPolygonとして復元できたもの
#
# hazard_current 側の既存ジオメトリには一切手を加えません（make_validもかけません）。
# 元の hazard_type もそのまま引き継ぎます。

hazard_current = gpd.GeoDataFrame(
    hazard_records.loc[hazard_records["kept_by_current"], ["hazard_type", "geometry"]]
    .reset_index(drop=True),
    crs="EPSG:4326",
)

repaired_source = hazard_records.loc[hazard_records["repaired_usable"]]
if len(repaired_source):
    repaired_additions = gpd.GeoDataFrame(
        {
            "hazard_type": repaired_source["hazard_type"].values,
            "geometry": make_valid_results.loc[repaired_source.index, "repaired_geometry"].values,
        },
        crs="EPSG:4326",
    )
    hazard_repaired = gpd.GeoDataFrame(
        pd.concat([hazard_current, repaired_additions], ignore_index=True), crs="EPSG:4326"
    )
else:
    repaired_additions = hazard_current.iloc[0:0]
    hazard_repaired = hazard_current.copy()

print(f"hazard_current : {len(hazard_current)}件（現行本体と同じ条件）")
print(f"hazard_repaired: {len(hazard_repaired)}件（うち復元分 {len(repaired_additions)}件を追加）")

if len(repaired_additions):
    print("\n復元分のハザード種別の内訳:")
    display(
        repaired_additions["hazard_type"]
        .value_counts()
        .rename_axis("hazard_type")
        .to_frame("件数")
    )
else:
    print("\n復元できたジオメトリが無いため、hazard_repaired は hazard_current と同じ内容です。")


In [ ]:
# ===== 7. 要支援者CSVによる地点判定への影響確認 =====
# 共通住民CSV（resident_id,address,latitude,longitude,geocode_status の5列）をアップロードし、
# 各要支援者地点について hazard_current と hazard_repaired の判定を比較します。
#
# 判定には現行 sheltermatch.ipynb と同じ intersects（空間インデックスで候補を絞ってから判定）を
# 使います。座標が使えない行は判定できないため、比較対象から外します。
#
# アップロードしたCSVの内容はこのNotebookの中だけで使います。個人情報を含むデータは、
# 利用を許可された環境でのみ扱ってください。


def read_csv_auto(file_bytes, label, dtype=None):
    """UTF-8(BOM付き) → CP932 → UTF-8 の順で読み込みを試みる（sheltermatch.ipynbと同じ方針）。"""
    last_error = None
    for encoding in ("utf-8-sig", "cp932", "utf-8"):
        try:
            return pd.read_csv(io.BytesIO(file_bytes), encoding=encoding, dtype=dtype)
        except (UnicodeDecodeError, UnicodeError) as error:
            last_error = error
            continue
    raise ValueError(f"[{label}] 文字コードを判定できませんでした。詳細: {last_error}")


def is_valid_coordinate(lat, lon):
    """緯度・経度が数値として有効な範囲かどうかを返す（sheltermatch.ipynbと同じ）。"""
    if pd.isna(lat) or pd.isna(lon):
        return False
    return (-90 <= lat <= 90) and (-180 <= lon <= 180)


def hazard_types_intersecting(geometry, hazard_area):
    """ジオメトリと交差するハザード区域を調べ、(該当したか, hazard_typeを;で連結した文字列) を返す。
    現行 sheltermatch.ipynb の hazard_types_intersecting() と同じ処理。"""
    if len(hazard_area) == 0:
        return False, ""
    hit_rows = hazard_area.sindex.query(geometry, predicate="intersects")
    hit_types = sorted(hazard_area["hazard_type"].iloc[hit_rows].unique())
    return (len(hit_types) > 0), ";".join(hit_types)


def compare_geometry(geometry):
    """1つのジオメトリについて、現行方式と修復方式の判定結果を比べる。"""
    current_hit, current_types = hazard_types_intersecting(geometry, hazard_current)
    repaired_hit, repaired_types = hazard_types_intersecting(geometry, hazard_repaired)
    return {
        "current_in_hazard": current_hit,
        "current_hazard_types": current_types,
        "repaired_in_hazard": repaired_hit,
        "repaired_hazard_types": repaired_types,
        "changed": (current_hit != repaired_hit) or (current_types != repaired_types),
    }


RESIDENT_REQUIRED_COLUMNS = ["resident_id", "address", "latitude", "longitude", "geocode_status"]

print("要支援者CSV（residents_geocoded.csv 等。5列の共通住民CSV）を選択してください。")
uploaded_residents = files.upload()

if not uploaded_residents:
    raise RuntimeError("要支援者CSVがアップロードされませんでした。ファイルを1つ選択してください。")

residents_filename = list(uploaded_residents.keys())[0]
residents = read_csv_auto(
    uploaded_residents[residents_filename], "要支援者一覧", dtype={"resident_id": str}
)

missing_columns = [c for c in RESIDENT_REQUIRED_COLUMNS if c not in residents.columns]
if missing_columns:
    raise ValueError(
        f"[要支援者一覧] 必須列が見つかりません: {', '.join(missing_columns)}\n"
        f"共通住民CSVは {','.join(RESIDENT_REQUIRED_COLUMNS)} の5列です。"
    )

resident_lat = pd.to_numeric(residents["latitude"], errors="coerce")
resident_lon = pd.to_numeric(residents["longitude"], errors="coerce")

resident_rows = []
for resident_id, lat, lon in zip(residents["resident_id"], resident_lat, resident_lon):
    if not is_valid_coordinate(lat, lon):
        continue
    row = {"resident_id": resident_id, "latitude": lat, "longitude": lon}
    row.update(compare_geometry(Point(lon, lat)))
    resident_rows.append(row)

resident_diff = pd.DataFrame(
    resident_rows,
    columns=["resident_id", "latitude", "longitude", "current_in_hazard", "current_hazard_types",
             "repaired_in_hazard", "repaired_hazard_types", "changed"],
)

print(f"\n'{residents_filename}' を読み込みました（{len(residents)}行）。")
print(f"座標が有効で判定できた要支援者: {len(resident_diff)}件")
print(f"判定結果が変わった要支援者    : {int(resident_diff['changed'].sum())}件")

DISPLAY_LIMIT = 50
display_columns = ["resident_id", "current_in_hazard", "current_hazard_types",
                   "repaired_in_hazard", "repaired_hazard_types", "changed"]

if len(resident_diff) > DISPLAY_LIMIT:
    print(f"\n先頭{DISPLAY_LIMIT}件を表示します（差分があった行は、このあとの「差分一覧」で全件表示します）。")
    display(resident_diff[display_columns].head(DISPLAY_LIMIT))
else:
    display(resident_diff[display_columns])


In [ ]:
# ===== 8. 候補避難所地点・直線への影響確認（BODIK API） =====
# BODIK Data APIから現行 sheltermatch.ipynb と同じ糸満市の避難所データを取得し、
# 要支援者ごとに geodesic でTOP_N=3の候補を算出したうえで、
#
#   - 候補避難所地点
#   - 要支援者→候補避難所を結ぶ直線
#
# について hazard_current と hazard_repaired の判定を比較します。
# 距離順位はハザード情報では変更しません（現行本体と同じく距離だけで決めます）。
#
# API取得に失敗した場合は、この比較だけをスキップします。ここまでのジオメトリ分析の結果は
# そのまま有効です。

TOP_N = 3
BODIK_BASE_URL = "https://data.bodik.jp"
BODIK_RESOURCE_ID = "3132a0a4-f522-4b2d-bf18-f106d8b3a5ae"  # 糸満市 指定緊急避難場所データセット


def fetch_shelters_from_bodik(base_url, resource_id, page_size=1000):
    """BODIKのCKAN Data API(datastore_search)から避難所データを全件取得する
    （現行 sheltermatch.ipynb と同じ取得方法）。"""
    endpoint = f"{base_url}/api/action/datastore_search"
    records = []
    offset = 0
    total = None

    while True:
        response = requests.get(
            endpoint,
            params={"resource_id": resource_id, "limit": page_size, "offset": offset},
            timeout=30,
        )
        response.raise_for_status()
        payload = response.json()
        if not payload.get("success"):
            raise RuntimeError("CKAN APIレスポンスが success=false を返しました。")

        result = payload.get("result")
        if result is None or "records" not in result:
            raise RuntimeError("CKAN APIレスポンスに result.records が含まれていません。")

        page_records = result["records"]
        records.extend(page_records)
        if total is None:
            total = result.get("total", len(page_records))
        offset += len(page_records)
        if len(page_records) == 0 or offset >= total:
            break

    if len(records) == 0:
        raise RuntimeError("BODIK APIの取得結果が0件でした。")
    return pd.DataFrame(records)


shelter_point_diff = pd.DataFrame(
    columns=["resident_id", "candidate_rank", "shelter_name", "current_in_hazard",
             "current_hazard_types", "repaired_in_hazard", "repaired_hazard_types", "changed"]
)
line_diff = pd.DataFrame(
    columns=["resident_id", "candidate_rank", "shelter_name", "current_in_hazard",
             "current_hazard_types", "repaired_in_hazard", "repaired_hazard_types", "changed"]
)
SHELTER_COMPARISON_DONE = False

try:
    shelters_raw = fetch_shelters_from_bodik(BODIK_BASE_URL, BODIK_RESOURCE_ID)
    print(f"BODIK APIから避難所一覧を{len(shelters_raw)}件取得しました。")
except Exception as error:
    print("BODIK APIから避難所一覧を取得できなかったため、候補避難所・直線の比較はスキップします。")
    print(f"（詳細: {error}）")
    print("ここまでのジオメトリ分析の結果はそのまま有効です。")
    shelters_raw = None

if shelters_raw is not None:
    shelters = shelters_raw.rename(columns={"名称": "name", "緯度": "latitude", "経度": "longitude"})
    missing_shelter_columns = [c for c in ("name", "latitude", "longitude") if c not in shelters.columns]
    if missing_shelter_columns:
        print(f"避難所データに必要な列がないため比較をスキップします: {', '.join(missing_shelter_columns)}")
    else:
        shelter_lat = pd.to_numeric(shelters["latitude"], errors="coerce")
        shelter_lon = pd.to_numeric(shelters["longitude"], errors="coerce")
        shelter_records = [
            (str(name), lat, lon)
            for name, lat, lon in zip(shelters["name"], shelter_lat, shelter_lon)
            if is_valid_coordinate(lat, lon)
        ]
        print(f"距離計算に使用する有効な避難所: {len(shelter_records)}件")

        # 同じ避難所地点は何度も候補になるため、地点の判定結果は1回だけ計算して使い回す。
        shelter_point_results = {}
        shelter_rows = []
        line_rows = []

        for resident_id, lat, lon in zip(
            resident_diff["resident_id"], resident_diff["latitude"], resident_diff["longitude"]
        ):
            candidates = sorted(
                (
                    (geodesic((lat, lon), (s_lat, s_lon)).meters, name, s_lat, s_lon)
                    for name, s_lat, s_lon in shelter_records
                ),
                key=lambda candidate: (candidate[0], candidate[1]),
            )[:TOP_N]

            for rank, (_distance_m, name, s_lat, s_lon) in enumerate(candidates, start=1):
                if (s_lat, s_lon) not in shelter_point_results:
                    shelter_point_results[(s_lat, s_lon)] = compare_geometry(Point(s_lon, s_lat))
                shelter_rows.append(
                    {"resident_id": resident_id, "candidate_rank": rank, "shelter_name": name,
                     **shelter_point_results[(s_lat, s_lon)]}
                )
                line_rows.append(
                    {"resident_id": resident_id, "candidate_rank": rank, "shelter_name": name,
                     **compare_geometry(LineString([(lon, lat), (s_lon, s_lat)]))}
                )

        shelter_point_diff = pd.DataFrame(shelter_rows, columns=shelter_point_diff.columns)
        line_diff = pd.DataFrame(line_rows, columns=line_diff.columns)
        SHELTER_COMPARISON_DONE = True

        print()
        print(f"候補避難所地点の判定: {len(shelter_point_diff)}件（延べ） / "
              f"変化あり {int(shelter_point_diff['changed'].sum())}件")
        print(f"直線交差の判定      : {len(line_diff)}件（延べ） / "
              f"変化あり {int(line_diff['changed'].sum())}件")


In [ ]:
# ===== 9. 差分一覧 =====
# 修復前後で判定結果が変わったものだけを、判定の種類（要支援者地点 / 候補避難所地点 / 直線交差）
# ごとに一覧表示します。差分が無い場合もその旨を明示します。

DIFF_SECTIONS = [
    ("要支援者地点", resident_diff, ["resident_id"]),
    ("候補避難所地点", shelter_point_diff, ["resident_id", "candidate_rank", "shelter_name"]),
    ("直線交差", line_diff, ["resident_id", "candidate_rank", "shelter_name"]),
]

changed_counts = {}
skipped_sections = []

for section_name, diff_frame, key_columns in DIFF_SECTIONS:
    if len(diff_frame) == 0:
        changed_counts[section_name] = 0
        skipped_sections.append(section_name)
        if section_name == "要支援者地点":
            print(f"【{section_name}】座標が有効な要支援者がいないため、比較できていません。")
        elif SHELTER_COMPARISON_DONE:
            print(f"【{section_name}】比較対象の候補がありませんでした。")
        else:
            print(f"【{section_name}】避難所データを取得できなかったため、比較していません"
                  "（判定差分が0件という意味ではありません）。")
        print()
        continue

    changed_rows = diff_frame[diff_frame["changed"]]
    changed_counts[section_name] = len(changed_rows)

    print(f"【{section_name}】判定件数 {len(diff_frame)}件 / 差分 {len(changed_rows)}件")
    if len(changed_rows) == 0:
        print("  修復前後で今回のテスト対象の判定差分は0件")
    else:
        display(
            changed_rows[
                key_columns
                + ["current_in_hazard", "current_hazard_types",
                   "repaired_in_hazard", "repaired_hazard_types"]
            ]
        )
    print()

if sum(changed_counts.values()) == 0:
    print("修復前後で今回のテスト対象の判定差分は0件")
if skipped_sections:
    print(f"※ 次の判定は比較できていません（0件という意味ではありません）: {', '.join(skipped_sections)}")


In [ ]:
# ===== 10. サマリ =====
# 調査結果をまとめ、本体へ修復処理を導入するか判断するための材料を表示します。
# このNotebookは「本体へ導入すべき」と結論づけません。判断は、この出力を見て行ってください。

total_records = len(hazard_records)
current_excluded = int((~hazard_records["kept_by_current"]).sum())
invalid_count = int((hazard_records["category"] == "invalid").sum())
repaired_usable_count = int(hazard_records["repaired_usable"].sum())
repaired_from_gc_count = int(hazard_records["repaired_from_gc"].sum())
unrepairable_count = invalid_count - repaired_usable_count

resident_changed = changed_counts.get("要支援者地点", 0)
shelter_changed = changed_counts.get("候補避難所地点", 0)
line_changed = changed_counts.get("直線交差", 0)
total_changed = resident_changed + shelter_changed + line_changed

print("調査対象ジオメトリ総数           : {}件".format(total_records))
print("現行方式で除外されるinvalid      : {}件".format(invalid_count))
print("make_validでPolygon系へ復元       : {}件".format(repaired_usable_count))
print("GeometryCollectionから復元可能    : {}件".format(repaired_from_gc_count))
print("復元不能                          : {}件".format(unrepairable_count))
print()
shelter_note = "" if SHELTER_COMPARISON_DONE else "（避難所データを取得できず未比較）"
print("要支援者地点の判定差分            : {}件".format(resident_changed))
print("候補避難所地点の判定差分          : {}件{}".format(shelter_changed, shelter_note))
print("直線交差判定の差分                 : {}件{}".format(line_changed, shelter_note))

print()
print("（参考）現行方式が除外するレコード全体: {}件".format(current_excluded))
print("  内訳  null: {}件 / empty: {}件 / invalid: {}件 / Polygon以外: {}件".format(
    int((hazard_records["category"] == "null").sum()),
    int((hazard_records["category"] == "empty").sum()),
    int((hazard_records["category"] == "invalid").sum()),
    int((hazard_records["category"] == "non_polygon").sum()),
))

if repaired_usable_count:
    repaired_area_total = float(hazard_records["repaired_area_m2"].sum())
    print("  復元されたポリゴンの面積合計: {:,.1f}平方メートル".format(repaired_area_total))

print()
print("=" * 60)
if repaired_usable_count == 0:
    conclusion = "修復可能な区域が実質的に無い"
    detail = (
        "is_valid=Falseのジオメトリから、区域判定に使えるPolygon/MultiPolygonを取り出せませんでした。"
        "現行方式の除外によって失われている区域情報は、今回のデータでは確認できません。"
    )
elif total_changed == 0:
    conclusion = "修復可能だが今回の判定には影響しない"
    detail = (
        f"make_valid()で{repaired_usable_count}件を復元できましたが、今回のテスト対象"
        "（要支援者地点・候補避難所地点・直線交差）では判定結果が1件も変わりませんでした。"
        "別の要支援者データでは変わる可能性があるため、判断する場合はこの点も踏まえてください。"
    )
else:
    conclusion = "修復可能な区域があり判定にも影響する"
    detail = (
        f"make_valid()で{repaired_usable_count}件を復元でき、今回のテスト対象でも"
        f"合計{total_changed}件の判定差分が出ました。上の差分一覧で、どの区域が効いているかを"
        "確認してください。"
    )

print(f"該当する判断: {conclusion}")
print("=" * 60)
print(detail)

print()
if not SHELTER_COMPARISON_DONE:
    print()
    print("※ 候補避難所地点・直線交差は、BODIK APIから避難所データを取得できなかったため")
    print("  比較していません。この2項目の『0件』は、差分が無かったという意味ではありません。")

print()
print("判断の際に確認してほしいこと:")
print("  - 復元されたポリゴンの面積（面積がほぼ0のものは、区域としての実体が無い断片の可能性）")
print("  - is_valid=Falseの理由（Self-intersection等）と、公式データが意図した区域との整合")
print("  - make_valid()の結果は元データの作成意図と一致する保証が無いこと")
print()
print("このNotebookは調査専用です。sheltermatch.ipynb の読込仕様・判定ロジックは変更していません。")
